In [1]:
import os
import json
import time
from datetime import datetime
import requests

# Set custom user agent header
headers = {"User-Agent": "TrendPulse/1.0"}

# Keywords to match each category
category_keywords = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Step 1: Fetch the list of top story IDs
top_stories_url = "https://hacker-news.firebaseio.com/v0/topstories.json"
response = requests.get(top_stories_url, headers=headers)
story_ids = response.json()[:500]  # Take the first 500 IDs

collected_stories = []
seen_ids = set()

# Step 2: Loop through each category
for category, keywords in category_keywords.items():
    print(f"Collecting stories for: {category}")
    category_count = 0

    # Check stories from the top stories list
    for story_id in story_ids:
        # Stop once we have 25 stories for this category
        if category_count >= 25:
            break

        # Avoid collecting the same story more than once
        if story_id in seen_ids:
            continue

        # Fetch story details
        item_url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
        try:
            item_res = requests.get(item_url, headers=headers, timeout=5)
            story = item_res.json()

            # Make sure the item exists and has a title
            if not story or "title" not in story:
                continue

            title_text = story["title"].lower()

            # Check if any keyword appears in the title
            is_match = False
            for word in keywords:
                if word in title_text:
                    is_match = True
                    break

            # If matched, save the required 7 fields
            if is_match:
                story_data = {
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                collected_stories.append(story_data)
                seen_ids.add(story_id)
                category_count += 1

        except Exception as e:
            # If an API request fails, print and continue to the next
            print(f"Error fetching story {story_id}: {e}")
            continue

    # Requirement: Wait 2 seconds between categories
    time.sleep(2)

# Step 3: Save to data/trends_YYYYMMDD.json
os.makedirs("data", exist_ok=True)
today_date = datetime.now().strftime("%Y%m%d")
file_path = f"data/trends_{today_date}.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(collected_stories, f, indent=4)

print(f"Collected {len(collected_stories)} stories. Saved to {file_path}")

Collected 97 stories. Saved to data/trends_20260901.json
